In [1]:
import pandas as pd
import psycopg2
import os

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

from dotenv import load_dotenv

load_dotenv("../.env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")



In [2]:
conn = psycopg2.connect(
    host="awesome-hw.sdsc.edu",
    port=5432,
    dbname="nourish",
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

def query_db(query):
    """Query the nourish database and return data as a `pd.Dataframe`"""
    try:
        with conn.cursor() as cursor:
            cursor.execute(query)
            columns = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()

        return pd.DataFrame(rows, columns=columns)
    except Exception as e:
        conn.rollback()
        raise e

query_db("SELECT 1")
print(f"Connection successful")

KeyboardInterrupt: 

In [3]:
def save_df_to_json(df: pd.DataFrame, filename: str):
    """Save a pandas DataFrame to a JSON file."""
    with open(f"../data/edges/{filename}", "w") as f:
        df.to_json(f, orient="records",index=False, indent=2)

    print(f"Data saved to data/edges/{filename}")

def read_node_json(filename: str) -> pd.DataFrame:
    """Read a JSON file into a pandas DataFrame."""
    df = pd.read_json(f"../data/nodes/{filename}")
    print(f"Data read from data/nodes/{filename}")
    return df

In [4]:
# Read node data to find IDs
state_df = read_node_json("state.json")
county_df = read_node_json("county.json")
city_df = read_node_json("city.json")
zipcode_df = read_node_json("zipcode.json")
community_df = read_node_json("community.json")
blockgroup_df = read_node_json("block_group.json")
business_df = read_node_json("business.json")
business_location_df = read_node_json("business_location.json")
zone_location_df = read_node_json("zone_location.json")
zone_type_df = read_node_json("zone_type.json")


Data read from data/nodes/state.json
Data read from data/nodes/county.json
Data read from data/nodes/city.json
Data read from data/nodes/zipcode.json
Data read from data/nodes/community.json
Data read from data/nodes/block_group.json
Data read from data/nodes/business.json
Data read from data/nodes/business_location.json
Data read from data/nodes/zone_location.json
Data read from data/nodes/zone_type.json


Relationship 1: `County` CONTAINED_IN `State`

In [121]:
df = query_db("""
    SELECT
        county AS entity1,
        'County' AS entitytype1,
        'CONTAINED_IN' AS predicate,
        state_name AS entity2,
        'State' AS entitytype2
    FROM county_neighborhoods         
""")
df["entity1"] = df["entity1"].map(dict(zip(county_df["name"], county_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(state_df["name"], state_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "county_contained_in_state.json")
df.head()

Data saved to data/edges/county_contained_in_state.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,1,County,CONTAINED_IN,1,State
1,2,County,CONTAINED_IN,1,State
2,3,County,CONTAINED_IN,1,State
3,4,County,CONTAINED_IN,1,State
4,5,County,CONTAINED_IN,1,State


Relationship 2: `City` CONTAINED_IN `County`

In [122]:
df = query_db("""
    SELECT
        city AS entity1,
        'City' AS entitytype1,
        'CONTAINED_IN' AS predicate,
        county AS entity2,
        'County' AS entitytype2
    FROM city_neighborhoods
""")
df["entity1"] = df["entity1"].map(dict(zip(city_df["name"], city_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(county_df["name"], county_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "city_contained_in_county.json")
df.head()

Data saved to data/edges/city_contained_in_county.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,16,City,CONTAINED_IN,37,County
1,25,City,CONTAINED_IN,37,County
2,24,City,CONTAINED_IN,37,County
3,21,City,CONTAINED_IN,37,County
4,68,City,CONTAINED_IN,37,County


Relationship 3: `Community` CONTAINED_IN `City`

In [123]:
df = query_db("""
    SELECT
        community AS entity1,
        'Community' AS entitytype1,
        'CONTAINED_IN' AS predicate,
        city AS entity2,
        'City' AS entitytype2
    FROM community_neighborhoods
""")
df["entity1"] = df["entity1"].map(dict(zip(community_df["name"], community_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(city_df["name"], city_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "community_contained_in_city.json")
df.head()

Data saved to data/edges/community_contained_in_city.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,10,Community,CONTAINED_IN,37,City
1,88,Community,CONTAINED_IN,37,City
2,308,Community,CONTAINED_IN,37,City
3,327,Community,CONTAINED_IN,8,City
4,328,Community,CONTAINED_IN,8,City


Relationship 4: `County` ADJACENT_TO `County`

In [124]:
df = query_db("""
    SELECT
        county AS entity1,
        'County' AS entitytype1,
        'ADJACENT_TO' AS predicate,
        unnest(neighboring_counties) AS entity2,
        'County' AS entitytype2
    FROM county_neighborhoods
""")
df["entity1"] = df["entity1"].map(dict(zip(county_df["name"], county_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(county_df["name"], county_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "county_adjacent_to_county.json")
df.head()


Data saved to data/edges/county_adjacent_to_county.json


,entity1,entitytype1,predicate,entity2,entitytype2
1,13,County,ADJACENT_TO,33,County
2,13,County,ADJACENT_TO,37,County
4,37,County,ADJACENT_TO,13,County
5,37,County,ADJACENT_TO,30,County
6,37,County,ADJACENT_TO,33,County


Relationship 5: `City` ADJACENT_TO `City`

In [125]:
df = query_db("""
    SELECT
        city AS entity1,
        'City' AS entitytype1,
        'ADJACENT_TO' AS predicate,
        unnest(neighboring_cities) AS entity2,
        'City' AS entitytype2
    FROM city_neighborhoods
""")
df["entity1"] = df["entity1"].map(dict(zip(city_df["name"], city_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(city_df["name"], city_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "city_adjacent_to_city.json")
df.head()


Data saved to data/edges/city_adjacent_to_city.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,16,City,ADJACENT_TO,12,City
1,16,City,ADJACENT_TO,17,City
2,16,City,ADJACENT_TO,21,City
3,25,City,ADJACENT_TO,8,City
4,25,City,ADJACENT_TO,19,City


Relationship 6: `Community` ADJACENT_TO `Community`

In [126]:
df = query_db("""
    SELECT
        community AS entity1,
        'Community' AS entitytype1,
        'ADJACENT_TO' AS predicate,
        unnest(neighboring_communities) AS entity2,
        'Community' AS entitytype2
    FROM community_neighborhoods
""")
df["entity1"] = df["entity1"].map(dict(zip(community_df["name"], community_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(community_df["name"], community_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "community_adjacent_to_community.json")
df.head()


Data saved to data/edges/community_adjacent_to_community.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,60,Community,ADJACENT_TO,44,Community
1,60,Community,ADJACENT_TO,45,Community
2,60,Community,ADJACENT_TO,56,Community
3,60,Community,ADJACENT_TO,61,Community
4,60,Community,ADJACENT_TO,67,Community


Relationship 7: `City` NEARBY `City`

In [127]:
df = query_db("""
    SELECT
        city AS entity1,
        'City' AS entitytype1,
        'NEARBY' AS predicate,
        unnest(nearby_cities) AS entity2,
        'City' AS entitytype2
    FROM city_neighborhoods
""")
df["entity1"] = df["entity1"].map(dict(zip(city_df["name"], city_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(city_df["name"], city_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "city_nearby_city.json")
df.head()

Data saved to data/edges/city_nearby_city.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,16,City,NEARBY,18,City
1,16,City,NEARBY,23,City
2,25,City,NEARBY,13,City
3,25,City,NEARBY,14,City
4,24,City,NEARBY,8,City


Relationship 8: `Community` NEARBY `Community`

In [128]:
df = query_db("""
    SELECT
        community AS entity1,
        'Community' AS entitytype1,
        'NEARBY' AS predicate,
        unnest(nearby_communities) AS entity2,
        'Community' AS entitytype2
    FROM community_neighborhoods
""")
df["entity1"] = df["entity1"].map(dict(zip(community_df["name"], community_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(community_df["name"], community_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "community_nearby_community.json")
df.head()


Data saved to data/edges/community_nearby_community.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,60,Community,NEARBY,6,Community
1,60,Community,NEARBY,28,Community
2,60,Community,NEARBY,31,Community
3,60,Community,NEARBY,40,Community
4,60,Community,NEARBY,46,Community


Relationship 9: `Community` OVERLAPS_WITH `Zipcode`

In [129]:
df = query_db("""
    SELECT
        community AS entity1,
        'Community' AS entitytype1,
        'OVERLAPS_WITH' AS predicate,
        unnest(zipcodes)::text AS entity2,
        'Zipcode' AS entitytype2
    FROM community_neighborhoods
""")
df["entity1"] = df["entity1"].map(dict(zip(community_df["name"], community_df["id"])))
df["entity2"] = df["entity2"].map(dict(zip(zipcode_df["zipcode"].astype(str), zipcode_df["id"])))
df = df[df["entity1"].notnull() & df["entity2"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "community_overlaps_zipcode.json")
df.head()


Data saved to data/edges/community_overlaps_zipcode.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,60,Community,OVERLAPS_WITH,69,Zipcode
1,60,Community,OVERLAPS_WITH,71,Zipcode
2,55,Community,OVERLAPS_WITH,76,Zipcode
3,55,Community,OVERLAPS_WITH,78,Zipcode
4,55,Community,OVERLAPS_WITH,79,Zipcode


Relationship 10: `Community` OVERLAPS_WITH `BlockGroup`

In [130]:
df = query_db("""
    SELECT
        name AS entity1,
        'Community' AS entitytype1,
        'OVERLAPS_WITH' AS predicate,
        unnest(intersected_block_groups)::text AS entity2,
        'BlockGroup' AS entitytype2
    FROM nourish_community_block_group_intersection
""")
df["entity1"] = df["entity1"].map(dict(zip(community_df["name"].str.upper(), community_df["id"])))
df = df[df["entity1"].notnull()]
df["entity1"] = df["entity1"].astype(int)
df["entity2"] = df["entity2"].astype(int)
save_df_to_json(df, "community_overlaps_blockgroup.json")
df.head()


Data saved to data/edges/community_overlaps_blockgroup.json


,entity1,entitytype1,predicate,entity2,entitytype2
33,15,Community,OVERLAPS_WITH,170391,BlockGroup
34,15,Community,OVERLAPS_WITH,170392,BlockGroup
35,15,Community,OVERLAPS_WITH,170394,BlockGroup
36,15,Community,OVERLAPS_WITH,170395,BlockGroup
37,15,Community,OVERLAPS_WITH,170501,BlockGroup


Relationship 11: `BusinessLocation` CONTAINED_IN `BlockGroup`

In [131]:
df = business_location_df
df = df[df["blockgroup"].notnull()]
df["entity1"] = df["id"]
df["entitytype1"] = "BusinessLocation"
df["predicate"] = "CONTAINED_IN"
df["entity2"] = df["blockgroup"].astype(int)
df["entitytype2"] = "BlockGroup"
df = df[["entity1", "entitytype1", "predicate", "entity2", "entitytype2"]]
save_df_to_json(df, "businesslocation_contained_in_blockgroup.json")
df.head()


Data saved to data/edges/businesslocation_contained_in_blockgroup.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,5,BusinessLocation,CONTAINED_IN,197021,BlockGroup
1,24,BusinessLocation,CONTAINED_IN,54023,BlockGroup
2,48,BusinessLocation,CONTAINED_IN,83241,BlockGroup
3,49,BusinessLocation,CONTAINED_IN,83462,BlockGroup
4,50,BusinessLocation,CONTAINED_IN,173061,BlockGroup


Relationship 11: `BusinessLocation` CONTAINED_IN `Zipcode`

In [132]:
df = business_location_df
df["entity1"] = df["id"]
df["entitytype1"] = "BusinessLocation"
df["predicate"] = "CONTAINED_IN"
df["entity2"] = df["zip"]
df["entitytype2"] = "Zipcode"
df = df[["entity1", "entitytype1", "predicate", "entity2", "entitytype2"]]

save_df_to_json(df, "businesslocation_contained_in_zipcode.json")
df.head()

Data saved to data/edges/businesslocation_contained_in_zipcode.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,5,BusinessLocation,CONTAINED_IN,92081,Zipcode
1,24,BusinessLocation,CONTAINED_IN,92101,Zipcode
2,48,BusinessLocation,CONTAINED_IN,92014,Zipcode
3,49,BusinessLocation,CONTAINED_IN,92121,Zipcode
4,50,BusinessLocation,CONTAINED_IN,92075,Zipcode


Relationship 12: `BusinessLocation` CONTAINED_IN `City`

In [133]:
df = read_node_json("business_location.json")
df["entity1"] = df["id"]
df["entitytype1"] = "BusinessLocation"
df["predicate"] = "CONTAINED_IN"
df["entity2"] = df["city"].map(dict(zip(city_df["name"], city_df["id"])))
df["entitytype2"] = "City"
df = df[df["entity2"].notnull()]
df["entity2"] = df["entity2"].astype(int)
df = df[["entity1", "entitytype1", "predicate", "entity2", "entitytype2"]]

save_df_to_json(df, "businesslocation_contained_in_city.json")
df.head()

Data read from data/nodes/business_location.json
Data saved to data/edges/businesslocation_contained_in_city.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,5,BusinessLocation,CONTAINED_IN,25,City
1,24,BusinessLocation,CONTAINED_IN,21,City
2,48,BusinessLocation,CONTAINED_IN,11,City
3,49,BusinessLocation,CONTAINED_IN,21,City
4,50,BusinessLocation,CONTAINED_IN,24,City


Relationship 13: `BusinessLocation` BELONGS_TO `Business`

In [134]:
df = read_node_json("business_location.json")
df["entity1"] = df["id"]
df["entitytype1"] = "BusinessLocation"
df["predicate"] = "BELONGS_TO"
df["entity2"] = df["name"].map(dict(zip(business_df["name"], business_df["id"])))
df["entitytype2"] = "Business"
df = df[["entity1", "entitytype1", "predicate", "entity2", "entitytype2"]]

save_df_to_json(df, "businesslocation_belongs_to_business.json")
df.head()


Data read from data/nodes/business_location.json
Data saved to data/edges/businesslocation_belongs_to_business.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,5,BusinessLocation,BELONGS_TO,1,Business
1,24,BusinessLocation,BELONGS_TO,2,Business
2,48,BusinessLocation,BELONGS_TO,3,Business
3,49,BusinessLocation,BELONGS_TO,4,Business
4,50,BusinessLocation,BELONGS_TO,5,Business


Relationship 14: `zone_location` BELONGS_TO `zone_type`

In [136]:
df = query_db("""
    SELECT
        id::TEXT AS entity1,
        'ZoneLocation' AS entitytype1,
        'BELONGS_TO' AS predicate,
        zone_name::TEXT AS entity2,
        'ZoneType' AS entitytype2
    FROM sandag_layer_zoning_base_sd_new
""")
df["entity2"] = df["entity2"].map(dict(zip(zone_type_df["name"], zone_type_df["id"])))
df = df[df["entity2"].notnull()]
save_df_to_json(df, "zonelocation_belongs_to_zonetype.json")
df.head()


Data saved to data/edges/zonelocation_belongs_to_zonetype.json


,entity1,entitytype1,predicate,entity2,entitytype2
0,1,ZoneLocation,BELONGS_TO,44,ZoneType
1,6,ZoneLocation,BELONGS_TO,44,ZoneType
2,7,ZoneLocation,BELONGS_TO,44,ZoneType
3,8,ZoneLocation,BELONGS_TO,44,ZoneType
4,9,ZoneLocation,BELONGS_TO,44,ZoneType


Relationship 15: `BusinessLocation` SHARED_REGION `BusinessLocation`

In [4]:
# We use a CTE 'target_subset' to pick the first 1000 businesses.
# We then find matches for ONLY those businesses against the full table.

subset_sql = """
WITH target_subset AS (
    SELECT * FROM ca_businesses_with_ai_franchise
    ORDER BY id 
    LIMIT 1000  -- <--- Adjust this number to change subset size
),
all_matches AS (
    -- 1. City Matches
    SELECT 
        t1.id AS entity1, 
        t2.id AS entity2, 
        'city' AS match_reason
    FROM target_subset t1
    JOIN target_subset t2 ON t1.city = t2.city AND t1.id != t2.id

    UNION ALL 

    -- 2. Zip Matches
    SELECT t1.id, t2.id, 'zip'
    FROM target_subset t1
    JOIN target_subset t2 ON t1.zip = t2.zip AND t1.id != t2.id

    UNION ALL 

    -- 3. Blockgroup Matches
    SELECT t1.id, t2.id, 'blockgroup'
    FROM target_subset t1
    JOIN target_subset t2 ON t1.blockgroup = t2.blockgroup AND t1.id != t2.id
)

-- Aggregate and Weight
SELECT 
    entity1,
    'BusinessLocation' AS entitytype1,
    entity2,
    'BusinessLocation' AS entitytype2,
    'shared_region' AS predicate,
    COUNT(*) AS weight,
    STRING_AGG(match_reason, ', ') AS shared_attributes
FROM all_matches
GROUP BY entity1, entity2;
"""

# Run the query using your existing function
print("Running query on subset...")
df_subset = query_db(subset_sql)

# Display results
print(f"Returned {len(df_subset)} connections.")
print(df_subset.head())

Running query on subset...
Returned 303590 connections.
   entity1       entitytype1  entity2       entitytype2      predicate  \
0        5  BusinessLocation      799  BusinessLocation  shared_region   
1        5  BusinessLocation     1274  BusinessLocation  shared_region   
2        5  BusinessLocation     2710  BusinessLocation  shared_region   
3        5  BusinessLocation     3285  BusinessLocation  shared_region   
4        5  BusinessLocation     4788  BusinessLocation  shared_region   

   weight shared_attributes  
0       2         city, zip  
1       2         city, zip  
2       2         zip, city  
3       2         city, zip  
4       2         city, zip  


In [8]:
df = read_node_json("business_location.json")
df = df[df["blockgroup"].notnull()]



Data read from data/nodes/business_location.json


In [ ]:
import duckdb
import os

# --- Configuration for NDP ---
# Adjust this based on your server instance size. 
# If you have 64GB RAM, set this to '50GB'.
MEMORY_LIMIT = '16GB' 
INPUT_FILE = '../data/nodes/business_location.json'
OUTPUT_FILE = '../data/edges/businesslocation_sharedregion_businesslocation.json'

print(f"Initializing DuckDB on NDP with {MEMORY_LIMIT} RAM limit...")

con = duckdb.connect()

# 1. Performance Tuning
con.execute(f"PRAGMA memory_limit='{MEMORY_LIMIT}';")
con.execute("SET preserve_insertion_order=false;") 
# Allow DuckDB to use all available cores
con.execute("SET threads=TO_DEFAULT;") 
# Ensure temp data goes to a location with space (optional, usually /tmp is fine)
# con.execute("SET temp_directory='/tmp/duckdb_temp';") 

# 2. The Original Single-Pass Query
# This is the fastest method because it streams data directly from JSON to JSON
sql_query = f"""
WITH source_data AS (
    SELECT id, city, zip, blockgroup 
    FROM read_json_auto('{INPUT_FILE}')
),
pairs AS (
    -- Blockgroup (Strongest)
    SELECT t1.id AS e1, t2.id AS e2, 'blockgroup' AS reason
    FROM source_data t1 
    JOIN source_data t2 ON t1.blockgroup = t2.blockgroup 
    WHERE t1.id < t2.id AND t1.blockgroup IS NOT NULL

    UNION ALL

    -- Zip
    SELECT t1.id, t2.id, 'zip'
    FROM source_data t1 
    JOIN source_data t2 ON t1.zip = t2.zip
    WHERE t1.id < t2.id AND t1.zip IS NOT NULL

    UNION ALL

    -- City (The heaviest join)
    SELECT t1.id, t2.id, 'city'
    FROM source_data t1 
    JOIN source_data t2 ON t1.city = t2.city
    WHERE t1.id < t2.id AND t1.city IS NOT NULL
)
SELECT 
    e1 AS entity1, 
    'BusinessLocation' AS entitytype1,
    e2 AS entity2, 
    'BusinessLocation' AS entitytype2,
    'shared_region' AS predicate,
    COUNT(*) AS weight,
    string_agg(reason, ', ') AS shared_attributes
FROM pairs
GROUP BY e1, e2
"""

print("Running High-Performance Join...")

# 3. Export to JSON Lines
# ARRAY FALSE is critical for Neo4j import compatibility
con.execute(f"""
    COPY ({sql_query}) 
    TO '{OUTPUT_FILE}' 
    (FORMAT JSON, ARRAY FALSE);
""")

print(f"Success! Output saved to {OUTPUT_FILE}")

Streaming data from disk to JSON (this may take a minute)...


OutOfMemoryException: Out of Memory Error: could not allocate block of size 256.0 KiB (5.5 GiB/5.5 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [10]:
business_location_df

NameError: name 'business_location_df' is not defined

In [7]:
pip install duckdb

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.8 MB 1.7 MB/s eta 0:00:08
   ---- ----------------------------------- 1.3/12.8 MB 2.0 MB/s eta 0:00:06
   ---- ----------------------------------- 1.6/12.8 MB 2.0 MB/s eta 0:00:06
   ------ --------------------------------- 2.1/12.8 MB 2.0 MB/s eta 0:00:06
   ------- -------------------------------- 2.4/12.8 MB 2.0 MB/s eta 0:00:06
   -------- ------------------------------- 2.9/12.8 MB 2.0 MB/s eta 0:00:05
   ---------- ----------------------------- 3.4/12.8 MB 2.0 MB/s eta 0:00:05
   ----------- ---------------------------- 3.7/12.8 MB 2.0 MB/s eta 0:00:05
   ------------ --------------------------- 3.9/12.8 MB 2.0 MB/s eta 0:00:05
   ------------- --